In [1]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

# 1. Descargamos los datos oficiales de casas en California
datos_casas = fetch_california_housing()

# 2. Los metemos en nuestra tabla mágica de Pandas (nuestra X con las pistas)
df_casas = pd.DataFrame(datos_casas.data, columns=datos_casas.feature_names)

# 3. Añadimos la columna de la respuesta final (nuestra y: el Precio)
# Nota: El precio original está medido en cientos de miles de dólares.
df_casas['Precio'] = datos_casas.target

# 4. Le damos un vistazo a nuestros ingredientes
df_casas.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Precio
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [2]:
# Escaneamos la tabla buscando celdas vacías y las contamos por columna
df_casas.isnull().sum()

MedInc        0
HouseAge      0
AveRooms      0
AveBedrms     0
Population    0
AveOccup      0
Latitude      0
Longitude     0
Precio        0
dtype: int64

In [4]:
# 1. Para X (Pistas): Eliminamos la columna 'Precio'
X = df_casas.drop('Precio', axis=1)

# 2. Para y (Objetivo): Nos quedamos ÚNICAMENTE con la columna 'Precio'
y = df_casas['Precio']

In [5]:
# Comprobamos las pistas (X)
print("--- MIS PISTAS (X) ---")
display(X.head(3))

# Comprobamos el objetivo (y)
print("\n--- MI RESPUESTA OBJETIVO (y) ---")
display(y.head(3))

--- MIS PISTAS (X) ---


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24



--- MI RESPUESTA OBJETIVO (y) ---


0    4.526
1    3.585
2    3.521
Name: Precio, dtype: float64

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

# 1. Separamos en 80% para estudiar y 20% para el examen
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Creamos el árbol especializado en predecir números (Regresión)
modelo_casas = DecisionTreeRegressor(max_depth=5, random_state=42)

# 3. ¡A estudiar! Entrenamos al modelo con los datos de entrenamiento
modelo_casas.fit(X_entrenamiento, y_entrenamiento)

DecisionTreeRegressor(max_depth=5, random_state=42)

In [7]:
# Pedimos al modelo que adivine los precios de las casas del examen
predicciones_casas = modelo_casas.predict(X_prueba)

# Vemos las primeras 5 predicciones
print(predicciones_casas[:5])

[1.16857267 1.27158128 3.19958174 2.20476301 1.64279735]


In [8]:
from sklearn.metrics import mean_absolute_error

# 1. Comparamos los precios reales (y_prueba) con las adivinanzas (predicciones_casas)
error_promedio = mean_absolute_error(y_prueba, predicciones_casas)

# 2. Multiplicamos por 100,000 porque los datos originales vienen en esa escala
error_en_dolares = error_promedio * 100000

print(f"El modelo se equivoca por un promedio de: ${error_en_dolares:,.2f} dólares por casa")

El modelo se equivoca por un promedio de: $52,225.93 dólares por casa


In [9]:
import pandas as pd

# Creamos una tabla para ver qué pistas tuvieron más peso para calcular el precio
importancias_casas = pd.DataFrame({
    'Pista': X_entrenamiento.columns,
    'Importancia': modelo_casas.feature_importances_
})

# Ordenamos la tabla de mayor a menor y mostramos los resultados
importancias_casas.sort_values(by='Importancia', ascending=False)

,Pista,Importancia
0,MedInc,0.771212
5,AveOccup,0.128407
1,HouseAge,0.041621
2,AveRooms,0.031261
6,Latitude,0.022049
4,Population,0.002485
7,Longitude,0.002097
3,AveBedrms,0.000869


In [13]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 1. Creamos el Bosque Aleatorio (por defecto, Scikit-Learn plantará 100 árboles)
# Le quitamos el límite de profundidad (max_depth) para que los árboles crezcan libres
modelo_bosque = RandomForestRegressor(random_state=42)

# 2. Entrenamos a los 100 árboles al mismo tiempo
modelo_bosque.fit(X_entrenamiento, y_entrenamiento)

# 3. Hacemos el examen final
predicciones_bosque = modelo_bosque.predict(X_prueba)

# 4. Calificamos el examen
error_bosque = mean_absolute_error(y_prueba, predicciones_bosque) * 100000

print(f"El nuevo margen de error con el Bosque es de: ${error_bosque:,.2f} dólares por casa")


El nuevo margen de error con el Bosque es de: $32,754.26 dólares por casa


In [23]:
import pandas as pd

# 1. Inventamos los datos de nuestra "casa de prueba"
# IMPORTANTE: Los valores van entre corchetes [ ] porque estamos creando una lista de 1 solo elemento
datos_nueva_casa = {
    'MedInc': [1.0],       # Ingreso medio del vecindario ($65,000 dólares)
    'HouseAge': [100.0],    # Casa relativamente nueva, 10 años
    'AveRooms': [3.0],     # Promedio de 6 cuartos en total
    'AveBedrms': [1.0],    # Promedio de 2 recámaras
    'Population': [1200.0],# 1200 personas viviendo en esa zona
    'AveOccup': [3.0],     # Promedio de 3 personas por casa
    'Latitude': [34.0],    # Coordenadas (ej. cerca de Los Ángeles)
    'Longitude': [-118.0]
}

# 2. Convertimos este expediente en una tabla de Pandas de 1 sola fila
nueva_casa = pd.DataFrame(datos_nueva_casa)

# 3. Le pedimos a nuestro Bosque Aleatorio que tase esta nueva casa
precio_predicho = modelo_bosque.predict(nueva_casa)

# 4. Mostramos el resultado (multiplicado por 100,000 para verlo en dólares reales)
print(f"🏠 El precio estimado por la IA para tu casa es: ${precio_predicho[0] * 100000:,.2f} dólares")

🏠 El precio estimado por la IA para tu casa es: $163,749.00 dólares
